In [ ]:
import asyncio
import importlib
import os
import sys
import threading


sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../")))

from pprint import pprint

import hololinked
import things

importlib.reload(hololinked)
importlib.reload(things)

<module 'things' from 'd:\\current projects\\hololinked-main\\tests\\things\\__init__.py'>

In [11]:
from things import OceanOpticsSpectrometer, TestThing

thing = TestThing(id="example-test")
spectrometer = OceanOpticsSpectrometer(id="example-spectrometer")

20-02-2026T18:38:21.320611Z [info     ] [THING] initialialised Thing class TestThing with id example-test Thing=TestThing thing_id=example-test
20-02-2026T18:38:21.322427Z [info     ] [THING] setup state machine, states=['DISCONNECTED', 'ON', 'FAULT', 'MEASURING', 'ALARM'], initial_state=DISCONNECTED Thing=OceanOpticsSpectrometer thing_id=example-spectrometer
20-02-2026T18:38:21.323063Z [info     ] [THING] initialialised Thing class OceanOpticsSpectrometer with id example-spectrometer Thing=OceanOpticsSpectrometer thing_id=example-spectrometer


In [12]:
pprint(thing.get_thing_model(ignore_errors=True).json(), indent=4)

20-02-2026T18:38:22.861693Z [error    ] [THING] Error while generating schema for base_property - WoT schema generator for this descriptor/property is not implemented. name base_property & type <class 'hololinked.core.property.Property'> Thing=TestThing thing_id=example-test
{   'actions': {   'action_echo': {'synchronous': True},
                   'execute_instruction': {   'description': 'executes '
                                                             'instruction '
                                                             'given by the '
                                                             'ASCII string '
                                                             'parameter '
                                                             "'command'.If "
                                                             'return data size '
                                                             'is greater than '
                                                    

In [ ]:
# not always True
assert thing.rpc_server is not None

pprint(
    thing.rpc_server.get_thing_description(
        thing, protocol="INPROC", ignore_errors=True,
    ),
    indent=4,
)

In [ ]:
TestThing.get_analogue_offset.to_affordance().json()

In [ ]:
TestThing.set_channel.to_affordance().json()

In [ ]:
TestThing.set_sensor_model.to_affordance().json()

In [ ]:
TestThing.set_channel_pydantic.to_affordance().json()

In [ ]:
from hololinked.utils import get_input_model_from_signature


get_input_model_from_signature(TestThing.set_channel_pydantic.obj)

In [ ]:
TestThing.start_acquisition.to_affordance().json()

In [ ]:
TestThing.execute_instruction.to_affordance().json()

In [ ]:
TestThing.data_point_event.to_affordance().json()

In [ ]:
TestThing.numpy_action.to_affordance().json()

In [13]:
import httpx
import json 

from hololinked.client.security import OAuthDirectAccessGrant, OAuth2Security

req_rep_timeout = httpx.Timeout(
    connect=10,
    read=10,
    write=10,
    pool=2,
)

req_rep_sync_client = httpx.Client(timeout=req_rep_timeout)
req_rep_async_client = httpx.AsyncClient(timeout=req_rep_timeout)

config = json.loads(open("oidc-config.json").read())

direct_access_grant = OAuthDirectAccessGrant(
    oidc_config_url=config["issuer"] + "/.well-known/openid-configuration",
    client_id=config["client_id"],
    client_secret=config["client_secret"],
    username=config["username"],
    password=config["password"],
)

In [15]:
from hololinked.client import ClientFactory

object_proxy_http = ClientFactory.http(
    "http://localhost:8080/test-thing/resources/wot-td", 
    ignore_TD_errors=True,
    security=direct_access_grant
)
object_proxy_http.td

{'context': ['https://www.w3.org/2022/wot/td/v1.1'],
 'id': 'test-thing',
 'title': 'TestThing',
 'description': 'A test thing with various API options for properties, actions and events that were collected from examples fromreal world implementations, testing, features offered etc.Add your own use case/snippets used in tests here as needed.',
 'properties': {'db_commit_number_prop': {'description': 'A fully editable number property to check commits to db on write operations',
   'default': 0,
   'type': 'number',
   'forms': [{'href': 'http://127.0.0.1:8080/test-thing/db-commit-number-prop',
     'op': 'readproperty',
     'htv:methodName': 'GET',
     'contentType': 'application/json'},
    {'href': 'http://127.0.0.1:8080/test-thing/db-commit-number-prop',
     'op': 'writeproperty',
     'htv:methodName': 'PUT',
     'contentType': 'application/json'}]},
  'db_init_int_prop': {'description': 'An integer property to check initialization from db',
   'default': 25,
   'type': 'integer

In [ ]:
import ssl 

from hololinked.client import ClientFactory


mqtt_ssl = ssl.create_default_context(ssl.Purpose.SERVER_AUTH)
if not os.path.exists("ca.crt"):
    raise FileNotFoundError(
        "CA certificate 'ca.crt' not found in current directory for MQTT TLS connection"
    )
mqtt_ssl.load_verify_locations(cafile="ca.crt")
mqtt_ssl.check_hostname = True
mqtt_ssl.verify_mode = ssl.CERT_REQUIRED
mqtt_ssl.minimum_version = ssl.TLSVersion.TLSv1_2

object_proxy_mqtt = ClientFactory.mqtt(hostname="localhost", port=8883, thing_id="example-test", ssl_context=mqtt_ssl, username="sampleuser", password="samplepass")


In [ ]:
from hololinked.client.factory import ClientFactory

object_proxy = ClientFactory.zmq(
    server_id="example-test",
    thing_id="example-test",
    protocol="IPC",
    ignore_TD_errors=True,
)
pprint(object_proxy.td, indent=4)

In [ ]:
def cb(value):
    print("event0", threading.get_ident(), value.data)


def cb1(value):
    print("event1 ", threading.get_ident(), value.data)


def cb2(value):
    print("event2 ", threading.get_ident(), value.data)


async def async_cb1(value):
    print("event1 ", asyncio.current_task().get_name(), value.data)


async def async_cb2(value):
    print("event2 ", asyncio.current_task().get_name(), value.data)


# object_proxy.subscribe_event("test_event", cb)
# object_proxy.subscribe_event("test_event", [cb1, cb2], concurrent=True)
object_proxy_mqtt.subscribe_event("test_event", [async_cb1, async_cb2], asynch=True)
# object_proxy.subscribe_event("test_event", [async_cb1, async_cb2], asynch=True, concurrent=True)

In [ ]:
object_proxy_mqtt.subscribe_event("test_event", cb)

In [ ]:
object_proxy_mqtt.subscribe_event(
    "test_event", [async_cb1, async_cb2], asynch=True, concurrent=True
)

In [ ]:
object_proxy_mqtt.subscribe_event(
    "test_binary_payload_event", callbacks=[cb1, cb2], concurrent=True
)

In [ ]:
object_proxy_http.push_events()

In [ ]:
object_proxy_http.push_events(event_name="test_binary_payload_event")

In [ ]:
object_proxy.unsubscribe_event("test_event")

In [ ]:
for i in range(5):
    print(object_proxy.read_property("db_init_int_prop"), i)

In [ ]:
object_proxy_mqtt.read_property("db_init_int_prop")

In [16]:
print(object_proxy_http.get_non_remote_number_prop())
object_proxy_http.set_non_remote_number_prop(20)
object_proxy_http.get_non_remote_number_prop()

5


20

In [23]:
object_proxy_http.get_transports()

AttributeError: 'RPCServer' object has no attribute 'ipc_server'

In [ ]:
# object_proxy.get_serialized_data()
object_proxy_zmq.get_serialized_data()

In [ ]:
val, binary_val = object_proxy_zmq.get_mixed_content_data()
print(val)
print(binary_val)

In [ ]:
# print(object_proxy.numpy_array_prop)
# print(type(object_proxy.numpy_array_prop))
print(object_proxy.numpy_action(object_proxy.numpy_array_prop))

In [ ]:
from hololinked.server.security import APIKeySecurity

auth = APIKeySecurity(name="test5", file="apikeys.json")
api_key = auth.create()

In [ ]:
from hololinked.server.security import APIKeySecurity

loaded_auth = APIKeySecurity(name="test5", file="apikeys.json")

In [ ]:
from datetime import datetime

loaded_auth.prehashed_apikey_expiry = datetime.fromisoformat(
    "2025-01-20T09:39:05.804308"
)

In [ ]:
loaded_auth.validate_input("...")

In [ ]:
from hololinked.server.security import KeycloakOIDCSecurity

auth = KeycloakOIDCSecurity(
    oidc_server_url="http://localhost:8080",
    oidc_realm="hololinked-test",
    oidc_client_id="device-server",
    verify_ssl=False,
    allowed_roles=["device-admin"],
)

token = ""
auth.validate_input(token)
userinfo = auth.userinfo(token)
print("userinfo", userinfo)

auth.user_has_role(userinfo)